# Building Energy Efficiency: Binary Classification (Assignment)

**Graded Assignment** - Train a **linear classifier** (single linear layer) for building energy efficiency classification.

## Tasks
- Implement neural networks via PyTorch
- Instantiate a linear classifier
- Solve supervised learning for building energy efficiency

In [1]:
import gzip
import pickle
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

## 1. Torch Dataset

In [2]:
path_to_embeddings = "../data/beec_dataset/embeddings/"
with open(path_to_embeddings + "embeddings_train.pkl.gz", "rb") as f:
    embeddings_train = pickle.loads(gzip.decompress(f.read()))
with open(path_to_embeddings + "embeddings_test.pkl.gz", "rb") as f:
    embeddings_test = pickle.loads(gzip.decompress(f.read()))
print(embeddings_train.shape, embeddings_test.shape)

torch.Size([4499, 384]) torch.Size([1500, 384])


In [3]:
path_to_labels = "../data/beec_dataset/labels/"
df_train = pd.read_csv(path_to_labels + "train.csv")
df_test = pd.read_csv(path_to_labels + "test.csv")
train_labels = torch.tensor(df_train["label"].tolist(), dtype=torch.long)
test_labels = torch.tensor(df_test["label"].tolist(), dtype=torch.long)

In [4]:
class Buildings(Dataset):
    def __init__(self, input_tensor, output_tensor):
        assert len(output_tensor) == len(input_tensor)
        self.input_tensor = input_tensor
        self.output_tensor = output_tensor
    def __len__(self):
        return len(self.output_tensor)
    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        return self.input_tensor[idx], self.output_tensor[idx]

train_dataset = Buildings(embeddings_train, train_labels)
test_dataset = Buildings(embeddings_test, test_labels)

## 2. Linear Classifier

Define a **single linear layer** connecting 384-dim embeddings to 2 output classes.

In [5]:
class LinearClassifier(nn.Module):
    def __init__(self, input_size, num_classes=2):
        super().__init__()
        self.linear = nn.Linear(input_size, num_classes)

    def forward(self, x):
        return self.linear(x)

model = LinearClassifier(input_size=384, num_classes=2)
print(model)

LinearClassifier(
  (linear): Linear(in_features=384, out_features=2, bias=True)
)


## 3. Training Loop

In [6]:
torch.manual_seed(42)
num_epochs, batch_size, learning_rate = 40, 32, 0.0001
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [7]:
def train_classifier():
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for embeds, labels in train_loader:
            outputs = model(embeds)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * embeds.size(0)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch [{epoch+1}/{num_epochs}], Training Loss: {running_loss/len(train_loader.dataset):.4f}")
        model.eval()
        total = correct = tp = fp = fn = 0
        with torch.no_grad():
            for embeds, labels in test_loader:
                _, predicted = torch.max(model(embeds), 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                tp += ((predicted == 1) & (labels == 1)).sum().item()
                fp += ((predicted == 1) & (labels == 0)).sum().item()
                fn += ((predicted == 0) & (labels == 1)).sum().item()
        acc = correct / total
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        print(f"Epoch [{epoch+1}/{num_epochs}], Test Accuracy: {acc:.2f}, Precision: {prec:.2f}, Recall: {rec:.2f}, F1: {f1:.2f}")
    return acc

accuracy = train_classifier()
print(f"Final accuracy: {accuracy:.2f}")

Epoch [1/40], Training Loss: 0.7628
Epoch [1/40], Test Accuracy: 0.49, Precision: 0.51, Recall: 0.51, F1: 0.51
Epoch [2/40], Training Loss: 0.7477
Epoch [2/40], Test Accuracy: 0.50, Precision: 0.52, Recall: 0.51, F1: 0.51


Epoch [3/40], Training Loss: 0.7344
Epoch [3/40], Test Accuracy: 0.50, Precision: 0.51, Recall: 0.51, F1: 0.51
Epoch [4/40], Training Loss: 0.7230
Epoch [4/40], Test Accuracy: 0.49, Precision: 0.51, Recall: 0.51, F1: 0.51


Epoch [5/40], Training Loss: 0.7130
Epoch [5/40], Test Accuracy: 0.50, Precision: 0.51, Recall: 0.52, F1: 0.52
Epoch [6/40], Training Loss: 0.7042
Epoch [6/40], Test Accuracy: 0.50, Precision: 0.52, Recall: 0.53, F1: 0.52


Epoch [7/40], Training Loss: 0.6968
Epoch [7/40], Test Accuracy: 0.50, Precision: 0.51, Recall: 0.52, F1: 0.52
Epoch [8/40], Training Loss: 0.6902
Epoch [8/40], Test Accuracy: 0.50, Precision: 0.52, Recall: 0.53, F1: 0.52


Epoch [9/40], Training Loss: 0.6846
Epoch [9/40], Test Accuracy: 0.50, Precision: 0.52, Recall: 0.53, F1: 0.52
Epoch [10/40], Training Loss: 0.6797
Epoch [10/40], Test Accuracy: 0.49, Precision: 0.51, Recall: 0.52, F1: 0.51


...

Epoch [31/40], Training Loss: 0.6502
Epoch [31/40], Test Accuracy: 0.49, Precision: 0.50, Recall: 0.48, F1: 0.49
Epoch [32/40], Training Loss: 0.6499
Epoch [32/40], Test Accuracy: 0.49, Precision: 0.50, Recall: 0.48, F1: 0.49


Epoch [33/40], Training Loss: 0.6497
Epoch [33/40], Test Accuracy: 0.49, Precision: 0.50, Recall: 0.48, F1: 0.49
Epoch [34/40], Training Loss: 0.6495
Epoch [34/40], Test Accuracy: 0.49, Precision: 0.50, Recall: 0.48, F1: 0.49


Epoch [35/40], Training Loss: 0.6494
Epoch [35/40], Test Accuracy: 0.49, Precision: 0.50, Recall: 0.48, F1: 0.49
Epoch [36/40], Training Loss: 0.6492
Epoch [36/40], Test Accuracy: 0.49, Precision: 0.51, Recall: 0.49, F1: 0.50


Epoch [37/40], Training Loss: 0.6491
Epoch [37/40], Test Accuracy: 0.49, Precision: 0.51, Recall: 0.49, F1: 0.50
Epoch [38/40], Training Loss: 0.6489
Epoch [38/40], Test Accuracy: 0.49, Precision: 0.51, Recall: 0.49, F1: 0.50


Epoch [39/40], Training Loss: 0.6489
Epoch [39/40], Test Accuracy: 0.49, Precision: 0.51, Recall: 0.49, F1: 0.50
Epoch [40/40], Training Loss: 0.6489
Epoch [40/40], Test Accuracy: 0.49, Precision: 0.50, Recall: 0.48, F1: 0.49
Final accuracy: 0.49
